In [2]:
!pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import mysql.connector
import pandas as pd

In [4]:
mydb = mysql.connector.connect(
    host="localhost",
    user="root",
    password=""
)

mycursor = mydb.cursor(buffered=True)

In [5]:
mycursor.execute("CREATE DATABASE tmdb_analytics")

DatabaseError: 1007 (HY000): Can't create database 'tmdb_analytics'; database exists

In [6]:
mycursor.execute("USE tmdb_analytics")

In [7]:
mycursor.execute("SHOW TABLES")

for table in mycursor:
    print(table)

('cast_cleaned',)
('crew_cleaned',)
('genres_cleaned',)
('movie_genres_cleaned',)
('movie_keywords_cleaned',)
('movies_cleaned',)


In [8]:
mycursor.execute("SELECT * FROM movies_cleaned")

result_movies = mycursor.fetchall()

In [9]:
df_movies = pd.DataFrame(
    result_movies,
    columns=mycursor.column_names
)

In [33]:
df_movies.head()

,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview
0,157336,Interstellar,en,2014-11-05,169,165000000.0,746606706.0,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...
1,27205,Inception,en,2010-07-15,148,160000000.0,839030630.0,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es..."
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1518815515.0,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1004558444.0,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...
4,19995,Avatar,en,2009-12-16,162,237000000.0,2923706026.0,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di..."


In [10]:
mycursor.execute("SELECT * FROM cast_cleaned")
result_cast = mycursor.fetchall()

df_cast = pd.DataFrame(
    result_cast,
    columns=mycursor.column_names
)

In [11]:
mycursor.execute("SELECT * FROM crew_cleaned")
result_crew = mycursor.fetchall()

df_crew = pd.DataFrame(
    result_crew,
    columns=mycursor.column_names
)

In [12]:
mycursor.execute("SELECT * FROM genres_cleaned")
result_genres = mycursor.fetchall()

df_genres = pd.DataFrame(
    result_genres,
    columns=mycursor.column_names
)

In [13]:
mycursor.execute("SELECT * FROM movie_genres_cleaned")
result_movie_genres = mycursor.fetchall()

df_movie_genres = pd.DataFrame(
    result_movie_genres,
    columns=mycursor.column_names
)

In [14]:
mycursor.execute("SELECT * FROM movie_keywords_cleaned")
result_movie_keywords = mycursor.fetchall()

df_movie_keywords = pd.DataFrame(
    result_movie_keywords,
    columns=mycursor.column_names
)

In [15]:
mycursor.execute("DESCRIBE movies_cleaned")
for col in mycursor.fetchall():
    print(col)

('movie_id', 'int(7)', 'YES', '', None, '')
('title', 'varchar(127)', 'YES', '', None, '')
('original_language', 'varchar(2)', 'YES', '', None, '')
('release_date', 'varchar(10)', 'YES', '', None, '')
('runtime', 'int(3)', 'YES', '', None, '')
('budget', 'varchar(18)', 'YES', '', None, '')
('revenue', 'varchar(18)', 'YES', '', None, '')
('popularity', 'decimal(7,4)', 'YES', '', None, '')
('vote_average', 'decimal(4,3)', 'YES', '', None, '')
('vote_count', 'int(5)', 'YES', '', None, '')
('status', 'varchar(8)', 'YES', '', None, '')
('overview', 'varchar(812)', 'YES', '', None, '')


In [ ]:
#Question1 - List all movies in the Action genre released after 2015, showing title and release date.

In [57]:
query1 = """
SELECT
    m.title,
    m.release_date
FROM movies_cleaned m
INNER JOIN movie_genres_cleaned mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres_cleaned g
    ON mg.genre_id = g.genre_id
WHERE
    g.genre_name = 'Action'
    AND YEAR(m.release_date) > 2015
"""

mycursor.execute(query1)

result1 = mycursor.fetchall()

df_actionmovie2015 = pd.DataFrame(
    result1,
    columns=mycursor.column_names
)

df_actionmovie2015.head(10)

,title,release_date
0,Deadpool,2016-02-09
1,Avengers: Infinity War,2018-04-25
2,Avengers: Endgame,2019-04-24
3,Captain America: Civil War,2016-04-27
4,Black Panther,2018-02-13
5,Doctor Strange,2016-10-25
6,Spider-Man: Homecoming,2017-07-05
7,Guardians of the Galaxy Vol. 2,2017-04-25
8,Spider-Man: No Way Home,2021-12-15
9,Suicide Squad,2016-08-03


In [45]:
df_actionmovie2015.to_csv(
    "action_movies_after_2015.csv",
    index=False
)

In [47]:
df_actionmovie2015.shape

(270, 2)

In [ ]:
#Question2 - Find the top 10 highest-grossing movies along with the genre(s) each belongs to.

In [17]:
query2 = """
SELECT
    m.title,
    m.revenue,
    GROUP_CONCAT(DISTINCT g.genre_name) AS genres
FROM movies_cleaned m
LEFT JOIN movie_genres_cleaned mg
    ON m.movie_id = mg.movie_id
LEFT JOIN genres_cleaned g
    ON mg.genre_id = g.genre_id
WHERE m.revenue > 0
GROUP BY m.movie_id
ORDER BY m.revenue DESC
LIMIT 10
"""

mycursor.execute(query2)
result2 = mycursor.fetchall()

df_top10revenuemovies = pd.DataFrame(result2, columns=mycursor.column_names)

df_top10revenuemovies.head(10)

,title,revenue,genres
0,Black Mass,99975678.0,"Crime,Drama,History"
1,Alpha,99630675.0,"Adventure,Drama"
2,Brazil,9953708.0,"Comedy,Science Fiction"
3,Jumanji: Welcome to the Jungle,995339117.0,"Action,Adventure,Comedy,Fantasy"
4,The Lucky One,99400000.0,"Drama,Romance"
5,Hellboy,99318987.0,"Action,Fantasy"
6,Shanghai Noon,99274467.0,"Adventure,Comedy,Western"
7,Portrait of a Lady on Fire,9923127.0,"Drama,Romance"
8,Never Let Me Go,9918093.0,"Drama,Romance,Science Fiction"
9,Blue Jasmine,99104804.0,Drama


In [59]:
df_top10revenuemovies.to_csv(
    "Top10_Revenue_movies.csv",
    index=False
)

In [ ]:
#Question3 - Calculate the average budget and average revenue for each genre, sorted by average revenue descending.

In [67]:
query3 = """
SELECT
    g.genre_name AS genre,
    ROUND(AVG(m.budget), 2) AS avg_budget,
    ROUND(AVG(m.revenue), 2) AS avg_revenue
FROM movies_cleaned m
INNER JOIN movie_genres_cleaned mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres_cleaned g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_id, g.genre_name
ORDER BY avg_revenue DESC
"""

mycursor.execute(query3)
result3 = mycursor.fetchall()

df_avgbudgetrevenue = pd.DataFrame(result3, columns=mycursor.column_names)

df_avgbudgetrevenue.head()

,genre,avg_budget,avg_revenue
0,Adventure,1.091516e+08,3.893034e+08
1,Animation,8.336061e+07,3.597435e+08
2,Family,8.458426e+07,3.430968e+08
3,Science Fiction,9.518393e+07,3.229880e+08
4,Action,9.314761e+07,2.993760e+08


In [68]:
df_avgbudgetrevenue.to_csv(
    "Average_budget_Revenue_genrewise.csv",
    index=False
)

In [ ]:
#Question4 - List the top 10 actors who have appeared in the most movies in the dataset.

In [71]:
query4 = """
SELECT
    actor_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM cast_cleaned
GROUP BY actor_name
ORDER BY movie_count DESC
LIMIT 10
"""

mycursor.execute(query4)
result4 = mycursor.fetchall()

df_top10actors = pd.DataFrame(result4, columns=mycursor.column_names)

df_top10actors.head(10)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Brad Pitt,38
2,Robert De Niro,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Mark Wahlberg,32
6,Scarlett Johansson,32
7,Willem Dafoe,32
8,Morgan Freeman,31
9,Tom Cruise,31


In [72]:
df_top10actors.to_csv(
    "Top_10_Actors.csv",
    index=False
)

In [ ]:
#Question 5 - Find all directors who have directed more than 3 movies, along with their movie count.

In [73]:
query5 = """
SELECT
    person_name AS director_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM crew_cleaned
WHERE job = 'Director'
GROUP BY person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC
"""

mycursor.execute(query5)
result5 = mycursor.fetchall()

df_directormorethan3films = pd.DataFrame(result5, columns=mycursor.column_names)

df_directormorethan3films.head(10)

,director_name,movie_count
0,Steven Spielberg,27
1,Ridley Scott,19
2,Tim Burton,19
3,Martin Scorsese,16
4,Michael Bay,15
5,Robert Zemeckis,15
6,Clint Eastwood,14
7,Robert Rodriguez,13
8,M. Night Shyamalan,13
9,Antoine Fuqua,12


In [74]:
df_directormorethan3films.to_csv(
    "Director_more_than_3films.csv",
    index=False
)

In [ ]:
#Question6 - List the top 10 most frequently used keywords across all movies.

In [76]:
query6 = """
SELECT
    keyword_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords_cleaned
GROUP BY keyword_name
ORDER BY movie_count DESC
LIMIT 10
"""

mycursor.execute(query6)
result6 = mycursor.fetchall()

df_top10keywords = pd.DataFrame(result6, columns=mycursor.column_names)

df_top10keywords.head(10)

,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


In [77]:
df_top10keywords.to_csv(
    "Top10_keywords.csv",
    index=False
)

In [ ]:
#Question7 - Find movies where the budget was above the overall average budget but the revenue was below the overall average revenue.

In [79]:
query7 = """
SELECT
    title,
    budget,
    revenue
FROM movies_cleaned
WHERE
    budget > (
        SELECT AVG(NULLIF(budget, 0))
        FROM movies_cleaned
    )
    AND
    revenue < (
        SELECT AVG(NULLIF(revenue, 0))
        FROM movies_cleaned
    )
ORDER BY budget DESC
"""

mycursor.execute(query7)
result7 = mycursor.fetchall()

df_highbudgetlowrevenue = pd.DataFrame(result7, columns=mycursor.column_names)

df_highbudgetlowrevenue.head(10)

,title,budget,revenue
0,The Lego Movie 2: The Second Part,99000000.0,199603202.0
1,The Road to El Dorado,95000000.0,76400000.0
2,Hollow Man,95000000.0,190213455.0
3,Lara Croft: Tomb Raider - The Cradle of Life,95000000.0,160099222.0
4,Seventh Son,95000000.0,114178613.0
5,The Expendables 3,95000000.0,214657577.0
6,Date Night,93214796.4159692,87595677.74492511
7,Tropic Thunder,92000000.0,195702811.0
8,Black Hawk Down,92000000.0,172989651.0
9,Percy Jackson: Sea of Monsters,90000000.0,199850315.0


In [80]:
df_highbudgetlowrevenue.to_csv(
    "High_Budget_Low_Revenue.csv",
    index=False
)

In [ ]:
#Question8a - List all actors who have appeared in a movie directed by a specific director -Christopher Nolan.

In [84]:
query8a = """
SELECT DISTINCT
    actor_name
FROM cast_cleaned c
INNER JOIN crew_cleaned cr
    ON c.movie_id = cr.movie_id
WHERE
    cr.job = 'Director'
    AND cr.person_name = 'Christopher Nolan'
ORDER BY actor_name
"""

mycursor.execute(query8a)
result8a = mycursor.fetchall()

df_nolanactors = pd.DataFrame(result8a, columns=mycursor.column_names)

df_nolanactors.head()

,actor_name
0,Aaron Eckhart
1,Al Pacino
2,Alon Aboutboul
3,Andy Serkis
4,Aneurin Barnard


In [83]:
df_nolanactors.to_csv(
    "Nolan_Movie_Actors.csv",
    index=False
)

In [ ]:
#Question8b - List all actors who have appeared in a movie directed by a specific director - James Cameron.

In [85]:
query8b = """
SELECT DISTINCT
    actor_name
FROM cast_cleaned c
INNER JOIN crew_cleaned cr
    ON c.movie_id = cr.movie_id
WHERE
    cr.job = 'Director'
    AND cr.person_name = 'James Cameron'
ORDER BY actor_name
"""

mycursor.execute(query8b)
result8b = mycursor.fetchall()

df_cameronactors = pd.DataFrame(result8b, columns=mycursor.column_names)

df_cameronactors.head()

,actor_name
0,Al Matthews
1,Arnold Schwarzenegger
2,Art Malik
3,Bernard Hill
4,Bess Motta


In [86]:
df_cameronactors.to_csv(
    "cameroan_Movie_Actors.csv",
    index=False
)

In [ ]:
#Question9 - Find the genre with the highest average rating, considering only genres with at least 20 movies.

In [92]:
query9 = """
SELECT
    g.genre_name AS genre,
    ROUND(AVG(m.vote_average), 2) AS avg_rating,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies_cleaned m
INNER JOIN movie_genres_cleaned mg
    ON m.movie_id = mg.movie_id
INNER JOIN genres_cleaned g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY avg_rating DESC
LIMIT 1
"""

mycursor.execute(query9)
result9 = mycursor.fetchall()

df_genrewithhighestavgrating = pd.DataFrame(result9, columns=mycursor.column_names)

df_genrewithhighestavgrating.head()

,genre,avg_rating,movie_count
0,War,7.36,85


In [97]:
df_genrewithhighestavgrating.to_csv(
    "Genre_with_highest_avg_rating.csv",
    index=False
)

In [ ]:
#Question10 - List the top 10 movies with the highest number of keywords tagged to them.

In [95]:
query10 = """
SELECT
    m.title,
    COUNT(DISTINCT mk.keyword_id) AS keyword_count
FROM movies_cleaned m
INNER JOIN movie_keywords_cleaned mk
    ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10
"""

mycursor.execute(query10)
result10 = mycursor.fetchall()

df_top10moviewithkeyword = pd.DataFrame(result10, columns=mycursor.column_names)

df_top10moviewithkeyword.head()

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54


In [98]:
df_top10moviewithkeyword.to_csv(
    "Top10_movies_with_keyword.csv",
    index=False
)